# 1.Runnable组件生命周期监听器

知识点讲解：with_listeners() 生命周期监听器与使用场景

1. 什么是生命周期监听器（Lifecycle Listener）
   - 在 Runnable 执行的关键节点挂载回调函数，实现执行过程的观测与干预
   - 属于面向切面编程（AOP）思想：不侵入业务逻辑即可插入横切关注点
   - 三个核心钩子：on_start（开始前）、on_end（成功结束后）、on_error（异常时）
   - with_listeners 是 Callbacks 机制的简约替代，方法更简洁、统一

2. 三个钩子的触发时机与互斥关系
   - on_start：组件即将执行前触发，此时 run_obj.outputs 为空
   - on_end：组件执行成功后触发，run_obj.outputs 已填充
   - on_error：组件执行抛异常时触发，run_obj.error 记录错误
   - 关键点：on_end 与 on_error 互斥，一次执行只触发其中之一

3. 回调函数的签名规范
   - 必须接收两个参数：(run_obj: Run, config: RunnableConfig)
   - 返回值被忽略，通常声明为 None
   - 回调自身抛异常会影响主流程，内部应做异常保护

4. Run 对象承载的关键信息
   - id：UUID，本次运行唯一标识，可串联分布式链路追踪
   - name / run_type：组件名称与运行类型（chain / llm / tool / retriever…）
   - start_time / end_time：起止时间，用于计算耗时
   - inputs / outputs：输入与输出数据；error：异常信息（仅 on_error 有值）
   - tags / metadata：附加标签与元数据

5. RunnableConfig 承载的运行时配置
   - configurable：运行时配置项（本例传入了自定义 name）
   - callbacks / tags / metadata / run_name / max_concurrency 等
   - 监听器可读取 configurable 中的业务上下文（user_id、trace_id 等）

6. 典型输出示例与观察点（课程示例）
   - on_start 阶段 run_obj.inputs 为 {'input': 2}，config 含 {'configurable': {'name': '慕小课'}}
   - on_end 阶段可读取完整 Run：id=UUID(...)、name='RunnableLambda'、run_type='chain'、inputs={'input':2}、outputs={'output': None}
   - 观察点：on_start 时 outputs 为空，on_end 时 outputs 才有值，印证钩子触发时机

7. 典型应用场景
   - 性能监控：用 end_time - start_time 统计各环节耗时
   - 日志审计：记录每次调用输入输出，满足合规与排障
   - 成本统计：on_end 累计 Token 消耗与费用
   - 异常告警：on_error 上报监控平台
   - 链路追踪：把 run_obj.id 写入 APM 系统

8. 与 Callbacks 机制的关系
   - with_listeners 是 Callbacks 的轻量语法糖，只暴露三个粗粒度钩子
   - 完整 BaseCallbackHandler 支持更细粒度事件（on_llm_new_token 等）
   - 简单监控用 with_listeners，复杂需求实现自定义 CallbackHandler

9. 底层实现原理（源码要点）
   - with_listeners() 将 on_start / on_end / on_error 合并进 config 的 callbacks，本质用 CallbackHandler 逻辑实现监听
   - 具体：创建一个 RootListenersTracer 对象包装三个回调，通过 RunnableBinding 的 config_factories 注入执行流程
   - _merge_configs 会把 config_factories 产出的回调与基础 config 合并

10. 数据流转与执行时序
    - chain.invoke(2, config={...})
    - → 触发 on_start(run_obj, config)，打印运行元信息
    - → 执行 lambda x: time.sleep(x)，阻塞 2 秒，返回 None
    - → 触发 on_end(run_obj, config)，此时可读取 outputs 与 end_time
    - → 若抛异常则改为触发 on_error，不会触发 on_end


In [ ]:
import time
from langchain_core.runnables import RunnableConfig
from langchain_core.runnables import RunnableLambda
from langchain_core.tracers.schemas import Run

In [ ]:
def on_start(run_obj: Run, config: RunnableConfig) -> None:
    """组件开始执行前的回调钩子
    触发时机：
        Runnable 即将执行业务逻辑之前，此时尚未产生输出
    参数：
        run_obj: Run - 本次运行的追踪对象，可读取的关键字段：
            - id: UUID - 运行唯一标识，可用于链路追踪
            - name: str - 组件名称（此处为 RunnableLambda）
            - run_type: str - 运行类型（此处为 chain）
            - start_time: datetime - 开始时间戳
            - inputs: dict - 输入数据（此处为 {"input": 2}）
            - outputs: dict - 此阶段为空或 None（尚未执行完成）
        config: RunnableConfig - 运行时配置字典，可读取：
            - configurable: dict - 自定义配置（此处含 {"name": "慕小课"}）
            - tags / metadata / callbacks 等
    返回：
        None - 返回值被框架忽略
    典型用途：
        记录请求开始日志、启动计时器、初始化追踪 span、校验输入合法性
    """
    print("on_start")
    print("run_obj:", run_obj)
    print("config:", config)
    print("============")

In [ ]:
def on_end(run_obj: Run, config: RunnableConfig) -> None:
    """组件执行成功后的回调钩子
    触发时机：
        Runnable 执行完成且未抛出异常，与 on_error 互斥
    参数：
        run_obj: Run - 本次运行的追踪对象，此阶段新增可用字段：
            - end_time: datetime - 结束时间戳，配合 start_time 可算出耗时
            - outputs: dict - 输出数据（此处 lambda 返回 None）
        config: RunnableConfig - 与 on_start 收到的是同一份配置
    返回：
        None
    典型用途：
        统计耗时（end_time - start_time）、记录成功日志、
        累计 Token 消耗与成本、上报监控指标、结束追踪 span
    """
    print("on_end")
    print("run_obj:", run_obj)
    print("config:", config)
    print("============")

In [ ]:
def on_error(run_obj: Run, config: RunnableConfig) -> None:
    """组件执行抛出异常时的回调钩子
    触发时机：
        Runnable 执行过程中抛出异常，与 on_end 互斥
    参数：
        run_obj: Run - 本次运行的追踪对象，此阶段关键字段：
            - error: str - 异常的字符串描述，用于定位问题原因
            - end_time: datetime - 异常发生的时间戳
            - inputs: dict - 导致失败的输入，便于复现问题
        config: RunnableConfig - 运行时配置
    返回：
        None
    注意：
        该回调不会阻止异常向上传播，异常仍会抛给调用方
        若需要吞掉异常，应使用 with_fallbacks 而非监听器
    典型用途：
        异常日志记录、上报监控告警平台、失败率统计、故障现场快照保存
    """
    print("on_error")
    print("run_obj:", run_obj)
    print("config:", config)
    print("============")

In [ ]:
runnable = RunnableLambda(lambda x: time.sleep(x)).with_listeners(
    on_start=on_start,
    on_end=on_end,
    on_error=on_error,
)
chain = runnable
chain.invoke(2, config={"configurable": {"name": "慕小课"}})